# SnapBoost Regression Example

This notebook demonstrates continuous target prediction with **SnapBoost** on the [Diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html) dataset.

In regression mode, SnapBoost minimizes mean squared error using the same heterogeneous learner pool.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from snapboost import SnapBoost

## Load and split the data

In [ ]:
X, y = load_diabetes(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Target range:     [{y.min():.1f}, {y.max():.1f}]")

## Train SnapBoost

In [ ]:
model = SnapBoost(
    num_iterations=100,
    learning_rate=0.1,
    p_tree=0.8,
    min_max_depth=4,
    max_max_depth=8,
    alpha=1.0,
    gamma=1.0,
    mode="regression",
    random_state=42,
    verbose=True,
)

model.fit(X_train, y_train)

## Evaluate on the test set

In [ ]:
r2 = model.score(X_test, y_test)
print(f"R²: {r2:.4f}")

rmse = model.evaluate(X_test, y_test)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.4f}")

## Predicted vs. actual

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")

ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title("SnapBoost Regression: Predicted vs. Actual")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## Residual distribution

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residuals, bins=20, edgecolor="black", alpha=0.7)
ax.axvline(0, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Residual (actual - predicted)")
ax.set_ylabel("Count")
ax.set_title("Test Set Residuals")
plt.tight_layout()
plt.show()